# Compare Original Sequential vs Score-Balanced ListT5 Tournament - Two Datasets

Notebook ini membandingkan:

1. `sequential`: tournament original-like ListT5, chunk sequential `0-4, 5-9, ...`;
2. `score_balanced`: extension grouping yang menyebarkan kandidat berdasarkan rank BM25: `1,21,41,61,81`, dst.

Dataset default:
- `trec-covid.jsonl`
- `scifact.jsonl`

Konfigurasi dasar:
- model: `Soyoung97/ListT5-base`
- BM25 top-100 candidates
- `topk=100`
- `listwise_k=5`
- `out_k=2`
- `rerank_topk=10`

Catatan: score-balanced memakai tournament machinery yang sama, tapi fungsi grouping per ronde diganti.


## 1. Kaggle Setup

Aktifkan GPU dan Internet di Kaggle. Jalankan cell ini dari runtime bersih.

In [ ]:
!python -V
!pip -q install -U pip setuptools wheel
!pip -q install sentencepiece jsonlines huggingface_hub

## 2. Clone Official ListT5 Repo

Repo ini berisi `FiDT5.py`, `run_listt5.py`, dan sample `trec-covid.jsonl`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/ListT5')
if not REPO_DIR.exists():
    !git clone https://github.com/soyoung97/ListT5.git /kaggle/working/ListT5

sys.path.append(str(REPO_DIR))
print(REPO_DIR)

## 3. Download Dataset TREC-COVID BM25 Top-100

Prioritas pertama memakai file bawaan repo. Kalau file tidak ada, download dari Hugging Face dataset official ListT5.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

DATASET_FILES = [
    'trec-covid.jsonl',
    'scifact.jsonl',
]

def resolve_dataset_path(filename):
    repo_dataset = REPO_DIR / filename
    if repo_dataset.exists():
        return repo_dataset
    return Path(hf_hub_download(
        repo_id='Soyoung97/beir-eval-bm25-top100',
        filename=filename,
        repo_type='dataset',
    ))

DATA_PATHS = {filename: resolve_dataset_path(filename) for filename in DATASET_FILES}
DATA_PATHS

## 4. Patch Kompatibilitas Kaggle Python 3.12

Official ListT5 ditulis untuk environment lama (`transformers==4.33.3`). Kaggle terbaru bisa error di `CheckpointWrapper` dan safetensors auto-conversion. Patch ini hanya untuk inference, tidak mengubah logika tournament.

In [ ]:
import torch
from transformers import T5Tokenizer
from FiDT5 import FiDT5, EncoderWrapper, CheckpointWrapper

# Patch EncoderWrapper for newer transformers.
EncoderWrapper.get_input_embeddings = lambda self: self.encoder.get_input_embeddings()
EncoderWrapper.set_input_embeddings = lambda self, new_embeddings: self.encoder.set_input_embeddings(new_embeddings)
EncoderWrapper.embed_tokens = property(lambda self: self.encoder.embed_tokens)

# Disable old checkpoint wrapper behavior during inference.
def checkpoint_wrapper_forward(self, *args, **kwargs):
    return self.module(*args, **kwargs)

CheckpointWrapper.forward = checkpoint_wrapper_forward

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 5. Load Pretrained ListT5-base

Menggunakan `use_safetensors=False` supaya Transformers tidak mencoba auto-convert checkpoint.

In [ ]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id='Soyoung97/ListT5-base',
    allow_patterns=['config.json', 'pytorch_model.bin', 'generation_config.json'],
)

tok = T5Tokenizer.from_pretrained('t5-base', legacy=False)
model = FiDT5.from_pretrained(model_dir, use_safetensors=False).to(device)
model.eval()

print('model loaded from:', model_dir)

## 6. Sanity Check Official Example

Output official README adalah format angka index, misalnya `[3 1 4 2 5]`. Bagian belakang dianggap lebih relevan oleh source original.

In [ ]:
texts = [
    'Query: When did Thomas Edison invent the light bulb?, Index: 1, Context: Lightning strike at Seoul National University',
    'Query: When did Thomas Edison invent the light bulb?, Index: 2, Context: Thomas Edison tried to invent a device for car but failed',
    'Query: When did Thomas Edison invent the light bulb?, Index: 3, Context: Coffee is good for diet',
    'Query: When did Thomas Edison invent the light bulb?, Index: 4, Context: KEPCO fixes light problems',
    'Query: When did Thomas Edison invent the light bulb?, Index: 5, Context: Thomas Edison invented the light bulb in 1879',
]

raw = tok(texts, return_tensors='pt', padding='max_length', max_length=128, truncation=True)
input_tensors = {
    'input_ids': raw['input_ids'].unsqueeze(0).to(device),
    'attention_mask': raw['attention_mask'].unsqueeze(0).to(device),
}

with torch.no_grad():
    output = model.generate(**input_tensors, max_length=7, return_dict_in_generate=True, output_scores=True)

print(tok.batch_decode(output.sequences, skip_special_tokens=True))

## 7. Original-Like Sequential Tournament Evaluator

Kode di bawah meniru bagian penting dari `run_listt5.py`:
- `get_rel_index`: ambil last `out_k` angka dari output model;
- `get_out_k`: sort index sebelum inference dan cache berdasarkan set index;
- `run_one_loop`: chunk sequential size 5, ambil top-2 dari tiap chunk, recursive aggregation;
- setelah top-1 ditemukan, original mengganti posisinya dengan dummy index, bukan mengecilkan list.


In [ ]:
import jsonlines
import math
import random
import statistics
import time
from collections import defaultdict
from tqdm.auto import tqdm

TOPK = 100
LISTWISE_K = 5
OUT_K = 2
RERANK_TOPK = 10
DUMMY_NUMBER = 21

MAX_INPUT_LENGTH_BY_DATASET = {
    'trec-covid.jsonl': 512,
    'scifact.jsonl': 512,
}
MAX_INPUT_LENGTH = 512  # overwritten per dataset during evaluation
MAX_GEN_LENGTH = LISTWISE_K + 2

num_forwards = 0

def make_listwise_text(question, ctxs):
    return [f'Query: {question}, Index: {i+1}, Context: {ctxs[i]}' for i in range(len(ctxs))]

def make_input_tensors(texts):
    raw = tok(
        texts,
        return_tensors='pt',
        padding='max_length',
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    return {
        'input_ids': raw['input_ids'].unsqueeze(0).to(device),
        'attention_mask': raw['attention_mask'].unsqueeze(0).to(device),
    }

def run_inference(input_tensors):
    global num_forwards
    with torch.no_grad():
        output = model.generate(
            **input_tensors,
            max_length=MAX_GEN_LENGTH,
            return_dict_in_generate=True,
            output_scores=True,
        )
    num_forwards += 1
    return output

def get_rel_index(output, k=OUT_K):
    gen_out = tok.batch_decode(output.sequences, skip_special_tokens=True)
    out_rel_indexes = []
    for iter_out in gen_out:
        out = iter_out.split(' ')[-k:]
        try:
            out_rel_index = [int(x) for x in out]
        except Exception:
            print('Bad generation output:', iter_out)
            out_rel_index = [1 for _ in range(k)]
        out_rel_indexes.append(out_rel_index)
    return out_rel_indexes

def get_leftover_idx(exclude, k, full_list, global_exclude):
    out = []
    i = 0
    exclude = list(set(exclude + global_exclude))
    allow_exclude = False
    if set(full_list) - set(exclude) == set():
        allow_exclude = True
    while len(out) != k:
        if i == len(full_list):
            i = 0
        if allow_exclude or (full_list[i] not in exclude):
            out.append(full_list[i])
        i += 1
    return out

def remove_duplicates(indexes):
    out = []
    for x in indexes:
        if x not in out:
            out.append(x)
    return out

def group2chunks(l, n=5):
    for i in range(0, len(l), n):
        yield l[i:i+n]

def get_out_k(question, full_ctxs, index, best_cache, use_cache=True, k=OUT_K):
    if len(set(index)) == 1:
        return index[:k]

    index = list(index)
    index.sort()  # original source canonicalizes index order before inference

    cache_key = tuple(set(index))
    if use_cache and best_cache.get(cache_key) is not None:
        return best_cache[cache_key][-k:]

    ctxs = [full_ctxs[x] for x in index]
    input_tensors = make_input_tensors(make_listwise_text(question, ctxs))
    output = run_inference(input_tensors)
    out_k_rel_index = get_rel_index(output, k=k)[0]

    try:
        out_k_def_index = [index[x - 1] for x in out_k_rel_index]
    except IndexError:
        out_k_def_index = index[-k:]

    best_cache[cache_key] = out_k_def_index
    return out_k_def_index

def run_one_loop(question, topk_ctxs, full_list_idx, best_cache, global_exclude):
    saved_index = []

    if (OUT_K * 2) > LISTWISE_K:
        full_list_idx = remove_duplicates(full_list_idx)

    grouped_list_idxs = list(group2chunks(full_list_idx, n=LISTWISE_K))

    # step 1: run chunkwise and select out_k from each chunk
    for cut_list in grouped_list_idxs:
        if len(cut_list) < LISTWISE_K:
            other_index = get_leftover_idx(cut_list, LISTWISE_K - len(cut_list), full_list_idx, global_exclude)
            saved_index += get_out_k(question, topk_ctxs, cut_list + other_index, best_cache, k=OUT_K)
        else:
            if len(set(cut_list)) == 1:
                saved_index.append(cut_list[0])
            else:
                saved_index += get_out_k(question, topk_ctxs, cut_list, best_cache, k=OUT_K)

    # step 2: aggregation
    if len(saved_index) < LISTWISE_K:
        other_index = get_leftover_idx(saved_index, LISTWISE_K - len(saved_index), full_list_idx, global_exclude)
        full_index = saved_index + other_index
        topk_out = get_out_k(question, topk_ctxs, full_index, best_cache, k=OUT_K)
        return topk_out[-1]
    elif len(saved_index) > LISTWISE_K:
        return run_one_loop(question, topk_ctxs, saved_index, best_cache, global_exclude)
    elif len(saved_index) == 1:
        return saved_index[0]
    else:
        return get_out_k(question, topk_ctxs, saved_index, best_cache, k=OUT_K)[-1]

def check_valid_list(full_list, global_exclude):
    for exc in global_exclude:
        while exc in full_list:
            exc_idx = full_list.index(exc)
            new_val = (exc + 1) % len(full_list)
            while new_val in global_exclude:
                new_val = (new_val + 1) % len(full_list)
            full_list[exc_idx] = new_val
    return full_list


## 8. Metrics Dan Data Helpers

In [ ]:
def dcg_at_k(relevances, k):
    return sum((2 ** rel - 1) / math.log2(i + 2) for i, rel in enumerate(relevances[:k]))

def ndcg_at_k(ranked_ids, qrels, k=10):
    rels = [qrels.get(doc_id, 0.0) for doc_id in ranked_ids[:k]]
    ideal = sorted(qrels.values(), reverse=True)[:k]
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(rels, k) / ideal_dcg

def load_jsonl(path):
    rows = []
    with jsonlines.open(path, 'r') as reader:
        for row in reader:
            rows.append(row)
    return rows

def get_qrels(instance):
    return {str(k): float(v) for k, v in instance.get('qrels', {}).items()}

def get_pid(doc):
    return str(doc.get('pid') or doc.get('docid'))

def get_top100_goldidx(instance):
    qrels = instance.get('qrels')
    if qrels is None:
        return []
    top100_pids = [get_pid(x) for x in instance['bm25_results'][:TOPK]]
    gold_pids = [x for x in qrels if qrels[x] != 0]
    top100_goldidx = []
    for pid in gold_pids:
        try:
            top100_goldidx.append(top100_pids.index(pid))
        except ValueError:
            continue
    return top100_goldidx

def bm25_ndcg(instance):
    qrels = get_qrels(instance)
    ranked_ids = [get_pid(x) for x in instance['bm25_results'][:TOPK]]
    return ndcg_at_k(ranked_ids, qrels, 10)


## 9. Rerank Satu Query Dengan Original-Like Sequential Tournament

In [ ]:
def sequential_groups(indexes, group_size=LISTWISE_K):
    return [list(indexes[i:i + group_size]) for i in range(0, len(indexes), group_size)]

def score_balanced_groups(indexes, group_size=LISTWISE_K):
    """Distribute candidates by BM25 rank across groups.

    For top-100 and group_size=5, this creates 20 groups:
    G1: 0,20,40,60,80
    G2: 1,21,41,61,81
    ...
    """
    indexes = sorted(indexes)
    n_groups = math.ceil(len(indexes) / group_size)
    groups = [[] for _ in range(n_groups)]
    for i, idx in enumerate(indexes):
        groups[i % n_groups].append(idx)
    return groups

def make_groups(indexes, strategy):
    if strategy == 'sequential':
        return sequential_groups(indexes, LISTWISE_K)
    if strategy == 'score_balanced':
        return score_balanced_groups(indexes, LISTWISE_K)
    raise ValueError(f'Unknown strategy: {strategy}')

def run_one_loop_strategy(question, topk_ctxs, full_list_idx, best_cache, global_exclude, strategy):
    saved_index = []

    if (OUT_K * 2) > LISTWISE_K:
        full_list_idx = remove_duplicates(full_list_idx)

    grouped_list_idxs = make_groups(full_list_idx, strategy)

    # step 1: run groupwise and select out_k from each group
    for cut_list in grouped_list_idxs:
        if len(cut_list) < LISTWISE_K:
            other_index = get_leftover_idx(cut_list, LISTWISE_K - len(cut_list), full_list_idx, global_exclude)
            saved_index += get_out_k(question, topk_ctxs, cut_list + other_index, best_cache, k=OUT_K)
        else:
            if len(set(cut_list)) == 1:
                saved_index.append(cut_list[0])
            else:
                saved_index += get_out_k(question, topk_ctxs, cut_list, best_cache, k=OUT_K)

    # step 2: aggregation, same logic as original
    if len(saved_index) < LISTWISE_K:
        other_index = get_leftover_idx(saved_index, LISTWISE_K - len(saved_index), full_list_idx, global_exclude)
        full_index = saved_index + other_index
        topk_out = get_out_k(question, topk_ctxs, full_index, best_cache, k=OUT_K)
        return topk_out[-1]
    elif len(saved_index) > LISTWISE_K:
        return run_one_loop_strategy(question, topk_ctxs, saved_index, best_cache, global_exclude, strategy)
    elif len(saved_index) == 1:
        return saved_index[0]
    else:
        return get_out_k(question, topk_ctxs, saved_index, best_cache, k=OUT_K)[-1]

def rerank_instance_strategy(instance, strategy='sequential', skip_no_candidate=False):
    question = instance['q_text']
    topk_docs = instance['bm25_results'][:TOPK]
    topk_ctxs = [f"{x.get('title', '')} {x.get('text', '')}".strip() for x in topk_docs]
    top100_goldidx = get_top100_goldidx(instance)

    if len(top100_goldidx) == 0 and skip_no_candidate:
        return [get_pid(x) for x in topk_docs], 0

    if len(topk_ctxs) <= 1:
        return [get_pid(x) for x in topk_docs], 0

    best_cache = {}
    global_exclude = []
    saved_topones = []
    full_list_idx = list(range(len(topk_ctxs)))
    start_forwards = num_forwards

    if len(full_list_idx) <= LISTWISE_K:
        out = get_out_k(question, topk_ctxs, full_list_idx, best_cache, use_cache=False, k=LISTWISE_K)
        while out:
            idx = out.pop()
            if idx not in saved_topones:
                saved_topones.append(idx)
    else:
        for _ in range(min(RERANK_TOPK, len(full_list_idx))):
            top1_def_idx = run_one_loop_strategy(question, topk_ctxs, full_list_idx, best_cache, global_exclude, strategy)
            top1_rel_idx = full_list_idx.index(top1_def_idx)
            global_exclude.append(top1_def_idx)

            if (len(full_list_idx) <= RERANK_TOPK) and (len(global_exclude) == len(full_list_idx)):
                saved_topones.append(top1_def_idx)
                break

            if (OUT_K * 2) > LISTWISE_K:
                full_list_idx = full_list_idx[:top1_rel_idx] + full_list_idx[top1_rel_idx:]
            else:
                full_list_idx[top1_rel_idx] = (top1_def_idx + DUMMY_NUMBER) % len(full_list_idx)

            full_list_idx = check_valid_list(full_list_idx, global_exclude)
            saved_topones.append(top1_def_idx)

    full_rank = saved_topones[:]
    for i in range(len(topk_ctxs)):
        if i not in saved_topones:
            full_rank.append(i)

    reranked_ids = [get_pid(topk_docs[rank_id]) for rank_id in full_rank]
    calls = num_forwards - start_forwards
    return reranked_ids, calls

rows_by_dataset = {filename: load_jsonl(path) for filename, path in DATA_PATHS.items()}

# Quick sanity check on the first query of the first dataset.
dataset_file = DATASET_FILES[0]
MAX_INPUT_LENGTH = MAX_INPUT_LENGTH_BY_DATASET[dataset_file]
instance = rows_by_dataset[dataset_file][0]
qrels = get_qrels(instance)

for strategy in ['sequential', 'score_balanced']:
    ranked_ids, calls = rerank_instance_strategy(instance, strategy=strategy)
    print('dataset:', dataset_file)
    print('strategy:', strategy)
    print('query:', instance['q_text'])
    print('BM25 nDCG@10:', bm25_ndcg(instance))
    print('ListT5 nDCG@10:', ndcg_at_k(ranked_ids, qrels, 10))
    print('calls:', calls)
    print('top10:', ranked_ids[:10])
    print()


## 10. Full TREC-COVID Evaluation

Cell ini menjalankan semua query TREC-COVID. Di Kaggle T4 bisa makan waktu cukup lama. Untuk debugging, set `MAX_QUERIES = 5` dulu. Untuk reproduksi full, pakai `MAX_QUERIES = None`.

In [ ]:
import pandas as pd
import statistics
import time
from tqdm.auto import tqdm

MAX_QUERIES_PER_DATASET = 10  # ganti 5/10/25; None = full dataset
SKIP_NO_CANDIDATE = False
STRATEGIES = ["sequential", "score_balanced"]

all_results = []
outputs = []
start = time.time()

for dataset_file in DATASET_FILES:
    MAX_INPUT_LENGTH = MAX_INPUT_LENGTH_BY_DATASET.get(dataset_file, 512)

    rows = rows_by_dataset[dataset_file]
    selected_rows = rows if MAX_QUERIES_PER_DATASET is None else rows[:MAX_QUERIES_PER_DATASET]
    dataset_name = dataset_file.replace(".jsonl", "")

    print("#" * 80)
    print("dataset:", dataset_file)
    print("queries:", len(selected_rows))
    print("max_input_length:", MAX_INPUT_LENGTH)

    for strategy in STRATEGIES:
        num_forwards = 0
        method_bm25 = []
        method_listt5 = []
        method_calls = []
        method_start = time.time()

        print("=" * 80)
        print("strategy:", strategy)

        for idx, instance in enumerate(tqdm(selected_rows, desc=f"{dataset_name}-{strategy}")):
            qrels = get_qrels(instance)

            ranked_ids, calls = rerank_instance_strategy(
                instance,
                strategy=strategy,
                skip_no_candidate=SKIP_NO_CANDIDATE,
            )

            bm25_score = bm25_ndcg(instance)
            listt5_score = ndcg_at_k(ranked_ids, qrels, 10)

            method_bm25.append(bm25_score)
            method_listt5.append(listt5_score)
            method_calls.append(calls)

            outputs.append({
                "dataset": dataset_name,
                "i": idx,
                "strategy": strategy,
                "q_text": instance["q_text"],
                "bm25_ndcg@10": bm25_score,
                "listt5_ndcg@10": listt5_score,
                "calls": calls,
                "top10": ranked_ids[:10],
            })

        row = {
            "dataset": dataset_name,
            "strategy": strategy,
            "queries": len(selected_rows),
            "bm25_mean_nDCG@10": statistics.mean(method_bm25) if method_bm25 else 0.0,
            "listt5_mean_nDCG@10": statistics.mean(method_listt5) if method_listt5 else 0.0,
            "mean_calls/query": statistics.mean(method_calls) if method_calls else 0.0,
            "total_forwards": num_forwards,
            "time_sec": round(time.time() - method_start, 2),
        }

        all_results.append(row)

        print("done:", dataset_name, strategy)
        print("BM25 mean nDCG@10:", row["bm25_mean_nDCG@10"])
        print("ListT5 mean nDCG@10:", row["listt5_mean_nDCG@10"])
        print("mean calls/query:", row["mean_calls/query"])
        print("time_sec:", row["time_sec"])

summary_df = pd.DataFrame(all_results)
outputs_df = pd.DataFrame(outputs)

print("total time sec:", round(time.time() - start, 2))

summary_df


## 11. Save Output

In [ ]:
df = pd.DataFrame(outputs)
out_csv = '/kaggle/working/listt5_seq_vs_score_balanced_2datasets_outputs.csv'
summary_csv = '/kaggle/working/listt5_seq_vs_score_balanced_2datasets_summary.csv'
df.to_csv(out_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
print(out_csv)
print(summary_csv)
summary_df
